# LaplacianNB sklearn Integration Tutorial

This notebook demonstrates how to use LaplacianNB with sklearn's ecosystem including pipelines, cross-validation, grid search, and the FingerprintTransformer.

## Setup and Imports

Let's import all necessary libraries for our sklearn integration examples.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# sklearn imports
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.base import clone
import matplotlib.pyplot as plt

# LaplacianNB imports
from laplaciannb import LaplacianNB_New, FingerprintTransformer, convert_fingerprints

# Set random seed for reproducibility
np.random.seed(42)

## Utility Functions

Define functions for molecular fingerprint generation.

In [ ]:
def get_molecular_fingerprints(smiles_list, n_bits=1024):
    """
    Convert SMILES to molecular fingerprints.
    
    Args:
        smiles_list: List of SMILES strings
        n_bits: Fingerprint size
        
    Returns:
        List of fingerprint sets
    """
    def get_fp(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if not mol:
            return set()
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
        fp = mfpgen.GetFingerprint(mol)
        return set(fp.GetOnBits())
    
    return [get_fp(smiles) for smiles in smiles_list]

## Create Synthetic Dataset

Let's create a larger synthetic dataset for demonstration.

In [ ]:
# Create synthetic molecular dataset
base_molecules = [
    # Alcohols (generally active)
    "CCO", "CCC", "CCCO", "CCCCO", "CCCCCO",
    "CC(C)O", "CCC(C)O", "CC(O)C",
    
    # Aromatics (generally inactive) 
    "c1ccccc1", "c1ccc(C)cc1", "c1ccc(CC)cc1", "c1ccc(O)cc1",
    "c1ccc(N)cc1", "c1ccc(Cl)cc1",
    
    # Carboxylic acids (generally active)
    "CC(=O)O", "CCC(=O)O", "CCCC(=O)O", "c1ccc(C(=O)O)cc1",
    "CC(C)C(=O)O", "CCCCC(=O)O",
    
    # Alkanes (generally inactive)
    "CC", "CCC", "CCCC", "CCCCC", "CCCCCC",
    "CC(C)C", "CC(C)CC", "CCC(C)C",
    
    # Alkenes (mixed activity)
    "C=C", "C=CC", "C=CCC", "CC=CC", "C=CC=C",
    
    # Ethers (mixed activity)
    "COC", "CCOC", "CCOCC", "c1ccc(OC)cc1"
]

# Define activity patterns (for demonstration)
activity_patterns = {
    # Alcohols -> active (1)
    0: [1, 1, 1, 1, 1, 1, 1, 1],
    # Aromatics -> inactive (0) 
    1: [0, 0, 0, 0, 0, 0],
    # Acids -> active (1)
    2: [1, 1, 1, 1, 1, 1],
    # Alkanes -> inactive (0)
    3: [0, 0, 0, 0, 0, 0, 0, 0],
    # Alkenes -> mixed
    4: [1, 0, 1, 0, 1],
    # Ethers -> mixed
    5: [0, 1, 0, 1]
}

# Build dataset
molecules = []
targets = []
molecule_types = []

type_names = ['Alcohols', 'Aromatics', 'Acids', 'Alkanes', 'Alkenes', 'Ethers']
start_idx = 0

for type_idx, (group_idx, activities) in enumerate(activity_patterns.items()):
    group_size = len(activities)
    group_molecules = base_molecules[start_idx:start_idx + group_size]
    
    molecules.extend(group_molecules)
    targets.extend(activities)
    molecule_types.extend([type_names[type_idx]] * group_size)
    
    start_idx += group_size

print(f"Created dataset with {len(molecules)} molecules")
print(f"Activity distribution: {np.bincount(targets)}")
print(f"Molecule types: {set(molecule_types)}")

In [ ]:
# Create DataFrame
df = pd.DataFrame({
    'smiles': molecules,
    'activity': targets,
    'molecule_type': molecule_types
})

# Display dataset summary
print("Dataset Summary:")
print(f"Total molecules: {len(df)}")
print(f"Active molecules: {sum(df['activity'])}")
print(f"Inactive molecules: {len(df) - sum(df['activity'])}")
print("\nMolecule types:")
print(df['molecule_type'].value_counts())

In [ ]:
# Show first few rows
df.head(10)

## Generate Molecular Fingerprints

Convert SMILES to molecular fingerprints for machine learning.

In [ ]:
# Generate fingerprints
print("Converting molecules to fingerprints...")
fingerprints = get_molecular_fingerprints(df['smiles'].tolist(), n_bits=1024)

# Add to dataframe
df['fingerprints'] = fingerprints

# Display fingerprint statistics
fp_sizes = [len(fp) for fp in fingerprints]
print(f"Fingerprint statistics:")
print(f"  Average bits per molecule: {np.mean(fp_sizes):.1f}")
print(f"  Min bits: {np.min(fp_sizes)}")
print(f"  Max bits: {np.max(fp_sizes)}")
print(f"  Std deviation: {np.std(fp_sizes):.1f}")

In [ ]:
# Plot fingerprint size distribution
plt.figure(figsize=(10, 6))
plt.hist(fp_sizes, bins=15, alpha=0.7, edgecolor='black')
plt.xlabel('Number of Bits Set')
plt.ylabel('Frequency')
plt.title('Distribution of Fingerprint Sizes')
plt.grid(True, alpha=0.3)
plt.show()

## Example 1: Basic sklearn Integration

Let's start with basic train/test split and evaluation.

In [ ]:
# Prepare data
X = fingerprints
y = df['activity'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Training set activity distribution: {np.bincount(y_train)}")
print(f"Test set activity distribution: {np.bincount(y_test)}")

In [ ]:
# Convert to sklearn format
X_train_sklearn = convert_fingerprints(X_train, n_bits=1024)
X_test_sklearn = convert_fingerprints(X_test, n_bits=1024)

print(f"Training matrix shape: {X_train_sklearn.shape}")
print(f"Test matrix shape: {X_test_sklearn.shape}")
print(f"Matrix format: {X_train_sklearn.format}")
print(f"Sparsity: {1 - X_train_sklearn.nnz / (X_train_sklearn.shape[0] * X_train_sklearn.shape[1]):.3f}")

In [ ]:
# Train classifier
clf = LaplacianNB_New(alpha=1.0)
clf.fit(X_train_sklearn, y_train)

# Evaluate
train_score = clf.score(X_train_sklearn, y_train)
test_score = clf.score(X_test_sklearn, y_test)

print(f"Training accuracy: {train_score:.3f}")
print(f"Test accuracy: {test_score:.3f}")

In [ ]:
# Detailed evaluation
y_pred = clf.predict(X_test_sklearn)
y_pred_proba = clf.predict_proba(X_test_sklearn)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Inactive', 'Active']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

## Example 2: Cross-Validation

Let's use cross-validation to get more robust performance estimates.

In [ ]:
# Convert all data to sklearn format
X_all = convert_fingerprints(X, n_bits=1024)
y_all = np.array(y)

print(f"Full dataset shape: {X_all.shape}")

In [ ]:
# Test different CV strategies
cv_strategies = {
    "5-fold CV": 5,
    "10-fold CV": 10,
    "Stratified 5-fold": StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    "Stratified 10-fold": StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
}

cv_results = {}

for name, cv in cv_strategies.items():
    scores = cross_val_score(clf, X_all, y_all, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:20s}: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")

In [ ]:
# Visualize CV results
plt.figure(figsize=(12, 6))
positions = range(len(cv_results))
bp = plt.boxplot([scores for scores in cv_results.values()], 
                 labels=list(cv_results.keys()),
                 patch_artist=True)

# Color the boxes
colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

plt.ylabel('Accuracy')
plt.title('Cross-Validation Results Comparison')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Example 3: Pipeline with Feature Selection

Let's create a pipeline that includes feature selection.

In [ ]:
# Create pipeline with feature selection
pipeline = Pipeline([
    ('feature_selection', SelectKBest(chi2, k=500)),
    ('classifier', LaplacianNB_New(alpha=1.0))
])

# Train pipeline
pipeline.fit(X_train_sklearn, y_train)
pipeline_score = pipeline.score(X_test_sklearn, y_test)

print(f"Pipeline test accuracy: {pipeline_score:.3f}")
print(f"Improvement over basic model: {pipeline_score - test_score:.3f}")

In [ ]:
# Cross-validate pipeline
pipeline_cv_scores = cross_val_score(pipeline, X_all, y_all, cv=5, scoring='accuracy')
basic_cv_scores = cross_val_score(clf, X_all, y_all, cv=5, scoring='accuracy')

print(f"Basic model CV accuracy: {basic_cv_scores.mean():.3f} (+/- {basic_cv_scores.std() * 2:.3f})")
print(f"Pipeline CV accuracy: {pipeline_cv_scores.mean():.3f} (+/- {pipeline_cv_scores.std() * 2:.3f})")

In [ ]:
# Analyze selected features
selector = pipeline.named_steps['feature_selection']
selected_features = selector.get_support()
feature_scores = selector.scores_

print(f"Selected {np.sum(selected_features)} out of {len(selected_features)} features")
print(f"Selected feature indices (first 20): {np.where(selected_features)[0][:20]}")
print(f"Top 10 feature scores: {np.sort(feature_scores)[-10:]}")

## Example 4: Grid Search Hyperparameter Tuning

Let's use grid search to optimize hyperparameters.

In [ ]:
# Define parameter grid
param_grid = {
    'feature_selection__k': [200, 500, 800],
    'classifier__alpha': [0.1, 1.0, 10.0]
}

print("Parameter grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

In [ ]:
# Perform grid search
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_sklearn, y_train)

In [ ]:
# Results
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")
print(f"Test score with best params: {grid_search.score(X_test_sklearn, y_test):.3f}")

# Show all results
results_df = pd.DataFrame(grid_search.cv_results_)
print("\nAll grid search results:")
print(results_df[['params', 'mean_test_score', 'std_test_score']].round(3))

## Example 5: FingerprintTransformer Pipeline

Now let's use the FingerprintTransformer to work directly with fingerprint sets.

In [ ]:
# Create pipeline with FingerprintTransformer
transformer_pipeline = Pipeline([
    ('fingerprints', FingerprintTransformer(n_bits=1024, output_format='csr')),
    ('feature_selection', SelectKBest(chi2, k=500)),
    ('classifier', LaplacianNB_New(alpha=1.0))
])

print("Pipeline steps:")
for step_name, step in transformer_pipeline.steps:
    print(f"  {step_name}: {type(step).__name__}")

In [ ]:
# Train on raw fingerprint sets (not pre-converted matrices)
transformer_pipeline.fit(X_train, y_train)
transformer_score = transformer_pipeline.score(X_test, y_test)

print(f"Transformer pipeline accuracy: {transformer_score:.3f}")

In [ ]:
# Cross-validate transformer pipeline
transformer_cv_scores = cross_val_score(transformer_pipeline, X, y, cv=5)
print(f"Transformer pipeline CV: {transformer_cv_scores.mean():.3f} (+/- {transformer_cv_scores.std() * 2:.3f})")

In [ ]:
# Grid search with transformer
transformer_param_grid = {
    'fingerprints__n_bits': [512, 1024],
    'fingerprints__output_format': ['csr', 'dense'],
    'feature_selection__k': [300, 500],
    'classifier__alpha': [0.5, 1.0]
}

transformer_grid = GridSearchCV(
    transformer_pipeline, 
    transformer_param_grid, 
    cv=3, 
    scoring='accuracy',
    verbose=1
)

transformer_grid.fit(X_train, y_train)

In [ ]:
print(f"Best transformer params: {transformer_grid.best_params_}")
print(f"Best transformer CV score: {transformer_grid.best_score_:.3f}")
print(f"Transformer test score: {transformer_grid.score(X_test, y_test):.3f}")

## Example 6: Model Comparison

Let's compare different alpha values and other configurations.

In [ ]:
# Test different alpha values
alpha_values = [0.01, 0.1, 1.0, 10.0, 100.0]
alpha_results = {}

for alpha in alpha_values:
    model = LaplacianNB_New(alpha=alpha)
    scores = cross_val_score(model, X_all, y_all, cv=5)
    alpha_results[alpha] = scores
    print(f"Alpha {alpha:6.2f}: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")

In [ ]:
# Visualize alpha comparison
plt.figure(figsize=(10, 6))
alphas = list(alpha_results.keys())
means = [scores.mean() for scores in alpha_results.values()]
stds = [scores.std() for scores in alpha_results.values()]

plt.errorbar(alphas, means, yerr=stds, marker='o', capsize=5, capthick=2)
plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('CV Accuracy')
plt.title('LaplacianNB Performance vs Alpha Parameter')
plt.grid(True, alpha=0.3)
plt.show()

## Example 7: Feature Importance Analysis

Let's analyze which molecular features are most important.

In [ ]:
# Train final model
final_model = LaplacianNB_New(alpha=1.0)
final_model.fit(X_all, y_all)

# Get feature importance (log probability differences)
feature_log_probs = final_model.feature_log_prob_
print(f"Feature log probabilities shape: {feature_log_probs.shape}")
print(f"Classes: {final_model.classes_}")

In [ ]:
# Calculate feature importance as log probability differences
class_0_probs = feature_log_probs[0]  # Inactive class
class_1_probs = feature_log_probs[1]  # Active class

# Difference (higher = more important for active class)
prob_diff = class_1_probs - class_0_probs

# Top features for each class
n_top = 10
top_inactive_features = np.argsort(prob_diff)[:n_top]  # Most negative
top_active_features = np.argsort(prob_diff)[-n_top:]   # Most positive

print(f"Top {n_top} features for INACTIVE class (bit indices): {top_inactive_features}")
print(f"Top {n_top} features for ACTIVE class (bit indices): {top_active_features}")

In [ ]:
# Visualize feature importance
plt.figure(figsize=(12, 8))

# Plot histogram of all feature differences
plt.subplot(2, 1, 1)
plt.hist(prob_diff, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Log Probability Difference (Active - Inactive)')
plt.ylabel('Number of Features')
plt.title('Distribution of Feature Importance Scores')
plt.grid(True, alpha=0.3)

# Plot top features
plt.subplot(2, 1, 2)
top_features = np.concatenate([top_inactive_features, top_active_features])
top_scores = prob_diff[top_features]
colors = ['red'] * n_top + ['green'] * n_top
labels = [f'Bit {i}' for i in top_features]

bars = plt.bar(range(len(top_features)), top_scores, color=colors, alpha=0.7)
plt.xlabel('Feature Index')
plt.ylabel('Importance Score')
plt.title(f'Top {n_top} Features for Each Class')
plt.xticks(range(len(top_features)), [f'{i}' for i in top_features], rotation=45)

# Add legend
import matplotlib.patches as mpatches
red_patch = mpatches.Patch(color='red', alpha=0.7, label='Inactive Class')
green_patch = mpatches.Patch(color='green', alpha=0.7, label='Active Class')
plt.legend(handles=[red_patch, green_patch])

plt.tight_layout()
plt.show()

## Example 8: Real-world Application Simulation

Let's simulate a real-world scenario with new molecule prediction.

In [ ]:
# Create some "new" molecules for prediction
new_molecules = [
    "CCCCCO",           # Long chain alcohol (probably active)
    "c1ccc(F)cc1",      # Fluorobenzene (probably inactive)
    "CCCCCC(=O)O",      # Hexanoic acid (probably active)
    "CCCCCCCC",         # Octane (probably inactive)
    "COc1ccccc1",       # Anisole (probably inactive)
    "CC(C)(C)O",        # tert-Butanol (probably active)
]

print("Predicting activity for new molecules:")
print("=" * 50)

In [ ]:
# Generate fingerprints for new molecules
new_fingerprints = get_molecular_fingerprints(new_molecules, n_bits=1024)
new_X = convert_fingerprints(new_fingerprints, n_bits=1024)

# Make predictions
new_predictions = final_model.predict(new_X)
new_probabilities = final_model.predict_proba(new_X)

# Display results
for i, smiles in enumerate(new_molecules):
    pred = new_predictions[i]
    prob_inactive, prob_active = new_probabilities[i]
    confidence = max(prob_inactive, prob_active)
    
    activity_label = "ACTIVE" if pred == 1 else "INACTIVE"
    print(f"{smiles:15s} -> {activity_label:8s} (confidence: {confidence:.3f})")
    print(f"{'':15s}    Probabilities: Inactive={prob_inactive:.3f}, Active={prob_active:.3f}")
    print()

## Summary

This tutorial demonstrated comprehensive sklearn integration with LaplacianNB:

### ✅ What we covered:

1. **Basic Integration**: Train/test splits and evaluation
2. **Cross-Validation**: Multiple CV strategies for robust evaluation  
3. **Pipelines**: Feature selection and preprocessing pipelines
4. **Grid Search**: Hyperparameter optimization
5. **FingerprintTransformer**: Direct integration with molecular fingerprints
6. **Model Comparison**: Alpha parameter optimization
7. **Feature Analysis**: Understanding important molecular features
8. **Real-world Application**: Predicting new molecule activities

### 🚀 Key Benefits:

- **sklearn Compatibility**: Full integration with sklearn ecosystem
- **Memory Efficiency**: Sparse matrix support for large fingerprints
- **Pipeline Support**: Easy integration with preprocessing and feature selection
- **Performance**: Fast training and prediction with molecular data
- **Flexibility**: Works with various fingerprint formats and sizes

### 🎯 Next Steps:

- Try with your own molecular datasets
- Experiment with different fingerprint types (ECFP, MACCS, etc.)
- Combine with other sklearn algorithms in ensembles
- Use in production pipelines for drug discovery